# SINA — Custom YOLOv8n Training (Colab)

Trains a **mobile, navigation-hazard** detector by merging **4 Roboflow datasets**
and exports a light **float16 TFLite (~6 MB)** that drops into the SINA app.

Datasets merged (by class name, so indices don't have to match):
- **OOD** `fpn/ood-pbnro` — 22 navigation classes (person, car, pole, curb, stairs…)
- **crosswalk** `tcc-xn3st/yolov8-crosswalk-detection` — adds `crosswalk`
- **stair-sidewalk** `elvis-cmqng/stair-sidewalk-yolov8` — more `stairs`
- **obstacles-for-blind** `obstacles-for-blind-zjnnn/obstacles-for-blind` — adds `door`, more stairs/people

**Before you start:** Runtime ▸ Change runtime type ▸ **T4 GPU**, and get a free
Roboflow API key at https://app.roboflow.com/settings/api .

Stays mobile-light: YOLOv8 **nano** + **imgsz 320** + **float16** export. Keep
those unchanged.


In [ ]:
!nvidia-smi

## 0. Install

In [ ]:
%pip install -q ultralytics ai-edge-litert roboflow pyyaml
import ultralytics, torch
ultralytics.checks()
print('CUDA available:', torch.cuda.is_available())

## 1. Download the 4 datasets

Paste your Roboflow API key below. Each download is wrapped in try/except — if one
slug/version is unavailable it's skipped and the rest still run (OOD is required).

In [ ]:
import re
from roboflow import Roboflow

rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_API_KEY")   # Settings > API key

def latest_version(project):
    nums = []
    for v in project.versions():
        m = re.search(r'(\d+)\s*$', str(getattr(v, 'version', v)))
        if m: nums.append(int(m.group(1)))
    return max(nums) if nums else 1

SOURCES = []   # list of (tag, location)
def dl(tag, ws, proj, ver=None):
    try:
        p = rf.workspace(ws).project(proj)
        v = ver or latest_version(p)
        print(f"Downloading {tag}: {ws}/{proj} v{v} ...")
        loc = p.version(v).download("yolov8").location
        SOURCES.append((tag, loc)); print("  ok ->", loc)
    except Exception as e:
        print(f"  SKIP {tag} ({ws}/{proj}): {e}")

dl("ood",   "fpn", "ood-pbnro", 1)
dl("cross", "tcc-xn3st", "yolov8-crosswalk-detection")
dl("stair", "elvis-cmqng", "stair-sidewalk-yolov8")
dl("blind", "obstacles-for-blind-zjnnn", "obstacles-for-blind")

print("\nDownloaded:", [t for t, _ in SOURCES])
assert any(t == "ood" for t, _ in SOURCES), "OOD failed — check your API key/quota."


## 2. Merge into one dataset (by class name)

Builds a unified class list = OOD's classes + `door` + `crosswalk`. Each source
label is normalised (`traffic_light`→`traffic light`) and mapped through a synonym
table; anything that doesn't match a unified class is dropped. Polygon
(segmentation) labels are converted to boxes. Prints what was **kept vs dropped**
so you can spot a useful class lost to a name mismatch (just add it to `SYN`).

In [ ]:
import os, glob, shutil, yaml
from collections import Counter

def names_of(loc):
    d = yaml.safe_load(open(os.path.join(loc, 'data.yaml')))
    n = d['names']
    return n if isinstance(n, list) else [n[i] for i in sorted(n)]

def norm(s): return str(s).replace('_', ' ').strip().lower()

# Map odd source names onto our canonical class names.
SYN = {
    'people':'person','pedestrian':'person','pedestrians':'person','human':'person',
    'doors':'door',
    'stair':'stairs','staircase':'stairs','steps':'stairs','stairscase':'stairs',
    'pedestrian crossing':'crosswalk','zebra crossing':'crosswalk','crossing':'crosswalk','cross walk':'crosswalk',
    'trafficlight':'traffic light','traffic signal':'traffic light','traffic-light':'traffic light',
    'stop':'stop sign','stopsign':'stop sign',
    'bike':'bicycle','cycle':'bicycle','bicyle':'bicycle',
    'motorbike':'motorcycle','motor':'motorcycle',
    'bin':'waste container','trash':'waste container','trash can':'waste container',
    'trashcan':'waste container','garbage':'waste container','dustbin':'waste container',
    'hydrant':'fire hydrant','poles':'pole',
}
def canon(s):
    s = norm(s); return SYN.get(s, s)

ood = dict(SOURCES)["ood"]
unified = [canon(n) for n in names_of(ood)]
for extra in ['door', 'crosswalk']:
    if extra not in unified: unified.append(extra)
seen = set(); unified = [x for x in unified if not (x in seen or seen.add(x))]
idx = {n: i for i, n in enumerate(unified)}
print("UNIFIED CLASSES (%d):" % len(unified))
for i, n in enumerate(unified): print(f"  {i}: {n}")

OUT = '/content/merged'
for s in ('train', 'valid'):
    os.makedirs(f'{OUT}/{s}/images', exist_ok=True)
    os.makedirs(f'{OUT}/{s}/labels', exist_ok=True)

kept, dropped = Counter(), Counter()
def ingest(tag, loc):
    names = names_of(loc)
    for split in os.listdir(loc):
        ip = os.path.join(loc, split, 'images'); lp = os.path.join(loc, split, 'labels')
        if not os.path.isdir(ip): continue
        dst = 'valid' if split in ('valid', 'val') else 'train'
        for img in glob.glob(ip + '/*.*'):
            base = os.path.splitext(os.path.basename(img))[0]
            ext = os.path.splitext(img)[1]
            nb = f'{tag}_{base}'
            shutil.copy(img, f'{OUT}/{dst}/images/{nb}{ext}')
            lf = os.path.join(lp, base + '.txt'); lines = []
            if os.path.exists(lf):
                for line in open(lf):
                    p = line.split()
                    if len(p) < 5: continue
                    src = names[int(float(p[0]))]; c = canon(src)
                    if c in idx:
                        coords = p[1:]
                        if len(coords) > 4:   # polygon -> bbox
                            xs = list(map(float, coords[0::2])); ys = list(map(float, coords[1::2]))
                            cx = (min(xs)+max(xs))/2; cy = (min(ys)+max(ys))/2
                            w = max(xs)-min(xs); h = max(ys)-min(ys)
                            lines.append(f"{idx[c]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                        else:
                            lines.append(' '.join([str(idx[c])] + coords[:4]))
                        kept[c] += 1
                    else:
                        dropped[norm(src)] += 1
            open(f'{OUT}/{dst}/labels/{nb}.txt', 'w').write('\n'.join(lines))

for tag, loc in SOURCES: ingest(tag, loc)

yaml.safe_dump({'path': OUT, 'train': 'train/images', 'val': 'valid/images',
                'nc': len(unified), 'names': unified}, open(f'{OUT}/data.yaml', 'w'))
DATA_YAML = f'{OUT}/data.yaml'

print("\nKEPT instances per class:")
for k, v in kept.most_common(): print(f"  {v:6d}  {k}")
if dropped:
    print("\nDROPPED (unmatched source names — add to SYN if you want them):")
    for k, v in dropped.most_common(): print(f"  {v:6d}  {k}")
n_tr = len(glob.glob(f'{OUT}/train/images/*')); n_va = len(glob.glob(f'{OUT}/valid/images/*'))
print(f"\nDATA_YAML = {DATA_YAML}")
print(f"train images: {n_tr}   valid images: {n_va}")


## 3. Train  (imgsz 320 → stays mobile-light, ~6 MB)

Transfer-learns from COCO `yolov8n.pt`. Drop `batch` to 32/16 on CUDA OOM. Keep
the tab active so Colab doesn't disconnect.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data=DATA_YAML,
    imgsz=320,                 # MUST equal kInputSize in the app
    epochs=60, batch=64, patience=15,
    name='sina', project='runs/detect', exist_ok=True,
    # cache='disk',            # uncomment to speed up if disk allows
)
BEST = 'runs/detect/sina/weights/best.pt'
print('best weights:', BEST)

## 4. Validate — overall + per-class mAP
Look at the per-class numbers: that tells you which classes (esp. curb / stairs) are actually usable.

In [ ]:
m = YOLO(BEST)
metrics = m.val(imgsz=320)
print('OVERALL  mAP50:', round(float(metrics.box.map50), 3),
      '  mAP50-95:', round(float(metrics.box.map), 3))
print('\nPer-class mAP@50 (worst first):')
names = m.names
try:
    ap50 = metrics.box.ap50            # AP@0.5 per present class
    idxs = metrics.box.ap_class_index  # which class ids those rows are
    rows = sorted(((names[int(c)], float(ap50[i])) for i, c in enumerate(idxs)), key=lambda r: r[1])
    for name, ap in rows: print(f"  {ap:5.3f}  {name}")
except Exception as e:
    print("  (per-class API differ:", e, ")")
    for c, v in enumerate(metrics.box.maps): print(f"  {round(float(v),3)}  {names[int(c)]}")

## 5. Export to TFLite (float16)

float16 keeps **float32 I/O** so the app's decoder works unchanged, and halves
size. Don't use int8 unless you add dequant to the Dart decoder.

In [ ]:
out = YOLO(BEST).export(format='tflite', imgsz=320, half=True)
TFLITE = str(out)
print('Exported:', TFLITE)

## 6. Verify I/O dtypes (must be float32 [1,320,320,3])

In [ ]:
import numpy as np
try:
    from ai_edge_litert.interpreter import Interpreter
except Exception:
    from tensorflow.lite import Interpreter
it = Interpreter(model_path=TFLITE); it.allocate_tensors()
i, o = it.get_input_details()[0], it.get_output_details()[0]
print('INPUT :', np.dtype(i['dtype']).name, list(i['shape']))
print('OUTPUT:', np.dtype(o['dtype']).name, list(o['shape']))
ok = (list(i['shape']) == [1, 320, 320, 3] and np.dtype(i['dtype']).name == 'float32')
assert ok, 'Input must be float32 [1,320,320,3] - re-export with imgsz=320, half=True'
print('OK - compatible with the SINA detector.')

## 7. Auto-generate the Dart constants + download the model

In [ ]:
names = YOLO(BEST).names
def n2(x): return str(x).replace('_', ' ').strip().lower()
labels = {int(i): n2(x) for i, x in names.items()}
tl = next((i for i, x in labels.items() if x == 'traffic light'), None)

print('// ---- paste into lib/constants/yolo_config.dart ----')
print('const int kTrafficLightClassId = %d;' % (tl if tl is not None else -1))
print('const Set<int> targetClassIds = {%s};' % ', '.join(str(i) for i in sorted(labels)))
print('const Map<int, String> cocoLabels = {')
for i in sorted(labels): print("  %d: '%s'," % (i, labels[i]))
print('};')
print('\n// Confirm labels.dart has fr/en/ar for each of these keys:')
for i in sorted(labels): print('//  ', labels[i])

from google.colab import files
files.download(TFLITE)

## 8. Plug into the SINA app

1. Save the downloaded file as **`sina_mobile/assets/models/yolov8n.tflite`** (overwrite).
2. Paste the Section-7 block into **`lib/constants/yolo_config.dart`** (replace the
   current `kTrafficLightClassId`, `targetClassIds`, `cocoLabels`).
3. `labels.dart` already has fr/en/ar for these classes. If Section 7 lists a key
   not yet there, add it.
4. `cd sina_mobile && flutter build apk --release`.

**Tips**
- Keep `imgsz=320` (training + export) — it must equal `kInputSize`.
- Trim noisy classes from `targetClassIds` (e.g. `cane`, maybe `tree`) if too chatty.
- Distance is bbox-area based, so treat curb / crosswalk as *presence* alerts.
- Re-run Section 6 after any re-export.
